# Phase 4: Customer Analysis

This notebook analyzes customer sales, profit, repeat purchases, revenue contribution, retention, and RFM segments.

## 1. Import libraries and load the cleaned dataset

In [ ]:
from pathlib import Path
import pandas as pd

current_folder = Path.cwd()

if current_folder.name == 'notebooks':
    project_folder = current_folder.parent
else:
    project_folder = current_folder

cleaned_file = project_folder / 'data' / 'processed' / 'cleaned_sales.csv'
sales = pd.read_csv(cleaned_file, parse_dates=['Order Date'])

print('Rows loaded:', len(sales))

## 2. Calculate customer-wise sales and profit

In [ ]:
customer_performance = (
    sales.groupby(['CustomerKey', 'Name'], as_index=False)
    .agg(
        Revenue=('Sales USD', 'sum'),
        Profit=('Profit USD', 'sum'),
        Orders=('Order Number', 'nunique'),
    )
    .sort_values('Revenue', ascending=False)
)

customer_performance[['Revenue', 'Profit']] = customer_performance[['Revenue', 'Profit']].round(2)
display(customer_performance.head())

## 3. Find repeat customers

In [ ]:
repeat_customers = customer_performance[customer_performance['Orders'] > 1].copy()

print('Total customers:', len(customer_performance))
print('Repeat customers:', len(repeat_customers))
display(repeat_customers.head())

## 4. Find the top 20 customers

In [ ]:
top_20_customers = customer_performance.head(20)
display(top_20_customers)

## 5. Find customers contributing the first 80% of revenue

In [ ]:
customer_performance['Cumulative Revenue'] = customer_performance['Revenue'].cumsum()
customer_performance['Cumulative Revenue %'] = (
    customer_performance['Cumulative Revenue']
    / customer_performance['Revenue'].sum()
    * 100
)

previous_cumulative_percent = customer_performance['Cumulative Revenue %'].shift(fill_value=0)
customers_contributing_80_percent = customer_performance[previous_cumulative_percent < 80].copy()

print('Customers contributing the first 80% of revenue:', len(customers_contributing_80_percent))
display(customers_contributing_80_percent)

## 6. Calculate customer retention rate

For this analysis, retention rate is the percentage of customers who placed more than one order.

In [ ]:
retention_rate = (len(repeat_customers) / len(customer_performance)) * 100
print(f'Customer retention rate: {retention_rate:.2f}%')

## 7. Calculate Recency, Frequency, and Monetary value

Recency is measured from one day after the latest order date in the dataset. Frequency is the number of distinct orders, and Monetary is total customer revenue.

In [ ]:
snapshot_date = sales['Order Date'].max() + pd.Timedelta(days=1)

customer_rfm = (
    sales.groupby(['CustomerKey', 'Name'], as_index=False)
    .agg(
        Last_Purchase_Date=('Order Date', 'max'),
        Frequency=('Order Number', 'nunique'),
        Monetary=('Sales USD', 'sum'),
        Profit=('Profit USD', 'sum'),
    )
)

customer_rfm['Recency'] = (snapshot_date - customer_rfm['Last_Purchase_Date']).dt.days
customer_rfm[['Monetary', 'Profit']] = customer_rfm[['Monetary', 'Profit']].round(2)

display(customer_rfm.head())

## 8. Create RFM scores

Each customer receives a score from 1 to 5 for Recency, Frequency, and Monetary value. A higher score is better.

In [ ]:
customer_rfm['R_Score'] = pd.qcut(
    customer_rfm['Recency'].rank(method='first'),
    5,
    labels=[5, 4, 3, 2, 1],
).astype(int)

customer_rfm['F_Score'] = pd.qcut(
    customer_rfm['Frequency'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5],
).astype(int)

customer_rfm['M_Score'] = pd.qcut(
    customer_rfm['Monetary'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5],
).astype(int)

customer_rfm['RFM_Score'] = (
    customer_rfm['R_Score']
    + customer_rfm['F_Score']
    + customer_rfm['M_Score']
)

## 9. Create customer segments

In [ ]:
def assign_customer_segment(rfm_score):
    if rfm_score >= 13:
        return 'Champions'
    elif rfm_score >= 10:
        return 'Loyal Customers'
    elif rfm_score >= 7:
        return 'Potential Loyalists'
    elif rfm_score >= 4:
        return 'At Risk'
    else:
        return 'Lost Customers'

customer_rfm['Customer Segment'] = customer_rfm['RFM_Score'].apply(assign_customer_segment)

segment_summary = (
    customer_rfm.groupby('Customer Segment', as_index=False)
    .agg(
        Customers=('CustomerKey', 'count'),
        Revenue=('Monetary', 'sum'),
        Profit=('Profit', 'sum'),
    )
    .sort_values('Revenue', ascending=False)
)

segment_summary[['Revenue', 'Profit']] = segment_summary[['Revenue', 'Profit']].round(2)
display(segment_summary)

## 10. Save the customer segmentation dataset

In [ ]:
output_file = project_folder / 'data' / 'processed' / 'customer_segments.csv'
customer_rfm.to_csv(output_file, index=False, date_format='%Y-%m-%d')

print('Customer segmentation dataset saved to:', output_file)

## 11. Display the Phase 4 answers

In [ ]:
print('PHASE 4 ANSWERS')
print('-' * 50)
print(f'Total customers: {len(customer_performance):,}')
print(f'Repeat customers: {len(repeat_customers):,}')
print(f'Customer retention rate: {retention_rate:.2f}%')
print(f'Customers contributing the first 80% of revenue: {len(customers_contributing_80_percent):,}')
print(f'Top customer: {top_20_customers.iloc[0]["Name"]}')
print(f'Top customer revenue: ${top_20_customers.iloc[0]["Revenue"]:,.2f}')
display(segment_summary)